
# 練習問題: omp simd でループをSIMD化する

## 目標

`#pragma omp simd` (Fortran では `!$omp simd`) を1つ挿入するだけで, コンパイラがループをSIMD命令 (`pd` 付きの packed double 命令) に変換することを体験する.

## 課題

`omp_simd.cpp` (または `omp_simd.f90`) は, saxpy/axpy と呼ばれる典型的な演算 `y[i] = a * x[i] + y[i]` を行う関数である.
配列 `x`, `y` が重なっている (エイリアスしている) 可能性をコンパイラが排除できないと, 安全のため自動ベクトル化を諦めてしまうことがある.

コメント `TODO` の指示に従って **OpenMP の指示行を1つ追加** し, ループがSIMD化されるようにせよ.

- C++: ループの直前に `#pragma omp simd` を1行加える.
- Fortran: doループの直前に `!$omp simd` を1行加える.

それ以外のコードを変更する必要はない.

## コンパイルと実行

`#pragma omp simd` を解釈させるには `-mp=multicore` が必要である (SIMD命令を使うだけなので1スレッドで動作する).

```
# C++
nvc++ -fast -mp=multicore -Mkeepasm -Minfo -Mneginfo -c omp_simd.cpp

# Fortran
nvfortran -fast -mp=multicore -Mkeepasm -Minfo -Mneginfo -c omp_simd.f90
```

- `-Mkeepasm` : 生成されたアセンブリ言語のファイル (`omp_simd.s`) を残す
- `-Minfo` / `-Mneginfo` : SIMD化に成功した・失敗したことを報告してくれる

生成されたアセンブリを確認する.

```
cat omp_simd.s
```

## 期待される結果

`#pragma omp simd` (`!$omp simd`) を追加すると, `omp_simd.s` の中に積和演算のSIMD命令 `vfmadd...pd` (または `vmulpd` + `vaddpd`) が現れる.
命令末尾の `pd` は _packed double precision_ の略で, _p_ (packed) がSIMD命令であることの証しである.
また `-Minfo` の出力に "Generated vector ..." のようなベクトル化成功のメッセージが現れることを確認せよ.

指示行を追加する前後でアセンブリ (`omp_simd.s`) や `-Minfo` の出力がどう変わるかを比べてみよ.



# 1. AIチューター
- 以下は必要に応じて実行（毎度実行する必要はない）


In [ ]:
import heytutor


## 1-1. 一般的な質問
- ChatGPTなどに聞くときのように自由に質問可能。
- ただし「答えを教えて」などは自制すること。


In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 1-2. この問題に関するヒント
- `{file:problem.md}` は上記の問題文に展開される。


In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}


## 1-3. いくつかの変数
* それぞれ以下のように展開される。

* `{file:FILENAME}` : _FILENAME_ の中身
* `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前の入力・出力, etc.

## 1-4. 困ったときのヘルプ
* コンパイル時や実行時のエラー直後に以下を実行するとエラーに関するヘルプが得られる。


In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:omp_simd.cpp}

コマンドと実行結果:
{bash[-1]}



## 1-5. フィードバック
* 答えが出た後も、無駄なところや、より良いやり方がないかを聞くことを推奨。
* 以下のファイル名は適宜書き換えよ (Fortranなら `.cpp` -> `.f90` とするなど)


In [ ]:
%%hey

フィードバックを下さい。

問題:
{file:problem.md}

私の答:
{file:omp_simd.cpp}


# 2. ジョブ投入ツール
- 以下を実行しておくと、`%%bash_submit_a` (Aquariousへのジョブ投入), `%%bash_submit_o` (Odyssey へのジョブ投入) が使えるようになる


In [ ]:
import wisteria_submit

# 3. C++ ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ omp_simd.cpp
/* y[i] = a * x[i] + y[i] (saxpy/axpy) を n 要素について行う */
void saxpy(long n, double a, double * x, double * y) {
  // TODO: 下の for ループの直前に #pragma omp simd を1行追加し, このループをSIMD化せよ.
  for (long i = 0; i < n; i++) {
    y[i] = a * x[i] + y[i];
  }
}

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore omp_simd.cpp -o omp_simd_cpp.exe

## 3-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./omp_simd_cpp.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a

./omp_simd_cpp.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./omp_simd_cpp.exe

## 3-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:omp_simd.cpp}


# 4. Fortran ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ omp_simd.f90
! y(i) = a * x(i) + y(i) (saxpy/axpy) を n 要素について行う
subroutine saxpy(n, a, x, y)
  implicit none
  integer(8), intent(in) :: n
  real(8), intent(in) :: a
  real(8), intent(in) :: x(n)
  real(8), intent(inout) :: y(n)
  integer(8) :: i
  ! TODO: 下の do ループの直前に !$omp simd を1行追加し, このループをSIMD化せよ.
  do i = 1, n
     y(i) = a * x(i) + y(i)
  end do
end subroutine saxpy

## 4-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore omp_simd.f90 -o omp_simd_f90.exe

## 4-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./omp_simd_f90.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a
./omp_simd_f90.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit

#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./omp_simd_f90.exe

## 4-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:omp_simd.f90}